# ARC-AGI-3 LLM Agent v7

Kombinacja najlepszych elementów:
- **v6.9**: Goal classifier + counter-picking (12 mechanizmów)
- **v7-abandon**: Early abandon dla martwych gier (0 poziomów po 60k tokenów)
- **Nowości**: Goal class retry, token budget tracking, adaptive exploration

**Architektura**: Qwen3.6-27B-FP8 via vLLM, tool-calling REPL z Python sandbox,
anti-loop detection, context management.

**Datasets**:
- `driessmit1/arc3-vllm-h100-wheelhouse-v3` (vLLM + torch wheels)
- `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot` (Qwen3.6-27B-FP8 model)

**Expected**: LB ~31-35 (vs v6.9 = 24.66, v7-abandon = ~31)


In [ ]:
import base64
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
NOTEBOOK_START = time.time()

os.environ["MPLBACKEND"] = "Agg"

cuda_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in [cuda_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if e
)

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

print(f"TRUE_SUBMISSION={TRUE_SUBMISSION}")
print("GPU check:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)


In [ ]:
# Install ARC runtime
print("Installing ARC runtime...")
subprocess.run([
    "pip", "install", "--quiet", "--no-deps",
    "/kaggle/input/arc3-vllm-h100-wheelhouse-v3/arcengine-1.0.0-cp311-cp311-linux_x86_64.whl"
], check=True)
print("✓ ARC runtime installed")


In [ ]:
# Install vLLM from wheelhouse
print("Installing vLLM from wheelhouse...")
wheelhouse = Path("/kaggle/input/arc3-vllm-h100-wheelhouse-v3")
wheels = list(wheelhouse.glob("*.whl"))
for whl in wheels:
    if whl.name.startswith("arcengine"):
        continue
    subprocess.run(["pip", "install", "--quiet", "--no-deps", str(whl)], check=True)
print(f"✓ Installed {len(wheels)} wheels")


In [ ]:
# Start vLLM server
model_path = "/kaggle/input/vrfai-qwen3-6-27b-fp8-hf-snapshot/Qwen3.6-27B-FP8"
print(f"Starting vLLM server with {model_path}...")

vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_path,
    "--served-model-name", "qwen3.6-27b-fp8",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--dtype", "bfloat16",
    "--max-model-len", "32768",
    "--gpu-memory-utilization", "0.90",
    "--tensor-parallel-size", "1",
    "--disable-log-requests",
]

server_log = WORKING / "vllm.log"
server_proc = subprocess.Popen(
    vllm_cmd,
    stdout=open(server_log, "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ, "VLLM_ATTENTION_BACKEND": "FLASHINFER"},
)
print(f"vLLM server PID: {server_proc.pid}")

# Wait for server
import time
for i in range(120):
    try:
        import urllib.request
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2):
            print("✓ vLLM server ready")
            break
    except Exception:
        time.sleep(1)
else:
    print("ERROR: vLLM server failed to start")
    sys.exit(1)


In [ ]:

# Deploy agent code
print("Deploying agent code...")

# Grid utilities
grid_utils_src = base64.b64decode("IiIiR3JpZCBmb3JtYXR0aW5nIGFuZCBzZWdtZW50YXRpb24gdXRpbGl0aWVzIGZvciBBUkMtQUdJLTMgTExNIGFnZW50LgoKRGVyaXZlZCBmcm9tIHRoZSBEdWNrIGhhcm5lc3MgKE1JVCBsaWNlbnNlLCBUdWZhIExhYnMpLgpTaW1wbGlmaWVkIGZvciB1c2Ugd2l0aCB0aGUgQVJDLUFHSS0zLUFnZW50cyBmcmFtZXdvcmsuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBTZXF1ZW5jZQoKIyBBUkMgY29sb3IgcGFsZXR0ZTogaW5kZXggLT4gc2luZ2xlLWNoYXIgbGFiZWwKQVJDX0NPTE9SX0NIQVJTID0gIld3Z0djQk1QUmJTWU9yTnAiCkFSQ19DT0xPUl9MRUdFTkQgPSAoCiAgICAiVz13aGl0ZSwgdz1saWdodCBncmF5LCBnPWdyYXksIEc9ZGFyayBncmF5LCBjPWNoYXJjb2FsLCBCPWJsYWNrLCAiCiAgICAiTT1tYWdlbnRhLCBQPXBpbmssIFI9cmVkLCBiPWJsdWUsIFM9c2t5IGJsdWUsIFk9eWVsbG93LCBPPW9yYW5nZSwgIgogICAgInI9ZGFyayByZWQsIE49bGlnaHQgZ3JlZW4sIHA9cHVycGxlIgopCgojIEFSQyBjb2xvciBwYWxldHRlOiBpbmRleCAtPiBSR0IKQVJDX0NPTE9SX01BUDogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50LCBpbnRdXSA9IHsKICAgIDA6ICgyNTUsIDI1NSwgMjU1KSwgMTogKDIwNCwgMjA0LCAyMDQpLCAyOiAoMTUzLCAxNTMsIDE1MyksCiAgICAzOiAoMTAyLCAxMDIsIDEwMiksIDQ6ICg1MSwgNTEsIDUxKSwgICAgNTogKDAsIDAsIDApLAogICAgNjogKDIyOSwgNTgsIDE2MyksICA3OiAoMjU1LCAxMjMsIDIwNCksIDg6ICgyNDksIDYwLCA0OSksCiAgICA5OiAoMzAsIDE0NywgMjU1KSwgIDEwOiAoMTM2LCAyMTYsIDI0MSksIDExOiAoMjU1LCAyMjAsIDApLAogICAgMTI6ICgyNTUsIDEzMywgMjcpLCAxMzogKDE0NiwgMTgsIDQ5KSwgIDE0OiAoNzksIDIwNCwgNDgpLAogICAgMTU6ICgxNjMsIDg2LCAyMTQpLAp9CgoKZGVmIGZvcm1hdF9ncmlkX2FzY2lpKGdyaWQ6IFNlcXVlbmNlW1NlcXVlbmNlW2ludF1dKSAtPiBzdHI6CiAgICAiIiJSZW5kZXIgYSBudW1lcmljIGdyaWQgYXMgQVNDSUkgYXJ0IHVzaW5nIEFSQyBjb2xvciBjaGFyYWN0ZXJzLiIiIgogICAgaWYgbm90IGdyaWQ6CiAgICAgICAgcmV0dXJuICIoZW1wdHkgZ3JpZCkiCiAgICBsaW5lcyA9IFtdCiAgICBmb3Igcm93IGluIGdyaWQ6CiAgICAgICAgY2hhcnMgPSBbXQogICAgICAgIGZvciB2IGluIHJvdzoKICAgICAgICAgICAgY2hhcnMuYXBwZW5kKEFSQ19DT0xPUl9DSEFSU1ttYXgoMCwgbWluKDE1LCBpbnQodikpKV0pCiAgICAgICAgbGluZXMuYXBwZW5kKCIiLmpvaW4oY2hhcnMpKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgpkZWYgZ3JpZF90b19wbmdfYnl0ZXMoZ3JpZDogU2VxdWVuY2VbU2VxdWVuY2VbaW50XV0sIHVwc2NhbGU6IGludCA9IDQpIC0+IGJ5dGVzOgogICAgIiIiUmVuZGVyIGEgbnVtZXJpYyBncmlkIGFzIGEgUE5HIGltYWdlIChubyBQSUwgZGVwZW5kZW5jeSDigJQgcHVyZSBieXRlcykuIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgICAgICAgaW1wb3J0IGlvCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcmV0dXJuIGIiIgogICAgCiAgICByb3dzID0gbGVuKGdyaWQpCiAgICBjb2xzID0gbWF4KChsZW4ocikgZm9yIHIgaW4gZ3JpZCksIGRlZmF1bHQ9MCkgaWYgcm93cyBlbHNlIDAKICAgIGlmIHJvd3MgPD0gMCBvciBjb2xzIDw9IDA6CiAgICAgICAgcmV0dXJuIGIiIgogICAgCiAgICBpbWFnZSA9IEltYWdlLm5ldygiUkdCIiwgKGNvbHMsIHJvd3MpLCBBUkNfQ09MT1JfTUFQWzBdKQogICAgcGl4ZWxzID0gaW1hZ2UubG9hZCgpCiAgICBmb3Igcm93X2lkeCwgcm93IGluIGVudW1lcmF0ZShncmlkKToKICAgICAgICBmb3IgY29sX2lkeCBpbiByYW5nZShjb2xzKToKICAgICAgICAgICAgdmFsdWUgPSByb3dbY29sX2lkeF0gaWYgY29sX2lkeCA8IGxlbihyb3cpIGVsc2UgMAogICAgICAgICAgICBwaXhlbHNbY29sX2lkeCwgcm93X2lkeF0gPSBBUkNfQ09MT1JfTUFQLmdldChpbnQodmFsdWUpLCBBUkNfQ09MT1JfTUFQWzBdKQogICAgCiAgICBpZiB1cHNjYWxlID4gMToKICAgICAgICBpbWFnZSA9IGltYWdlLnJlc2l6ZSgoY29scyAqIHVwc2NhbGUsIHJvd3MgKiB1cHNjYWxlKSwgSW1hZ2UuUmVzYW1wbGluZy5ORUFSRVNUKQogICAgCiAgICBidWZmZXIgPSBpby5CeXRlc0lPKCkKICAgIGltYWdlLnNhdmUoYnVmZmVyLCBmb3JtYXQ9IlBORyIpCiAgICByZXR1cm4gYnVmZmVyLmdldHZhbHVlKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFNlZ21lbnRhdGlvbiAoY29ubmVjdGVkLWNvbXBvbmVudCBhbmFseXNpcykKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCl9PUlRIID0gKCgtMSwgMCksICgxLCAwKSwgKDAsIC0xKSwgKDAsIDEpKQpfQ1cgPSAoKC0xLCAtMSksICgtMSwgMCksICgtMSwgMSksICgwLCAxKSwgKDEsIDEpLCAoMSwgMCksICgxLCAtMSksICgwLCAtMSkpCl9DV19JTkRFWCA9IHtvZmY6IGkgZm9yIGksIG9mZiBpbiBlbnVtZXJhdGUoX0NXKX0KCgpkZWYgX3RyYWNlX291dGVyX2NvbnRvdXIoY2VsbHMsIHN0YXJ0KToKICAgICIiIk1vb3JlLW5laWdoYm91ciB0cmFjZSBvZiBvdXRlciBwZXJpbWV0ZXIsIGNsb2Nrd2lzZS4iIiIKICAgIGlmIGxlbihjZWxscykgPT0gMToKICAgICAgICByZXR1cm4gW3N0YXJ0XQogICAgY29udG91ciA9IFtzdGFydF0KICAgIGIgPSBzdGFydAogICAgcHJldiA9IChzdGFydFswXSwgc3RhcnRbMV0gLSAxKQogICAgc2Vjb25kID0gTm9uZQogICAgZm9yIF8gaW4gcmFuZ2UoOCAqIGxlbihjZWxscykgKyAxNik6CiAgICAgICAgaWR4ID0gX0NXX0lOREVYWyhwcmV2WzBdIC0gYlswXSwgcHJldlsxXSAtIGJbMV0pXQogICAgICAgIG54dCA9IE5vbmUKICAgICAgICBmb3IgayBpbiByYW5nZSgxLCA5KToKICAgICAgICAgICAgb2ZmID0gX0NXWyhpZHggKyBrKSAlIDhdCiAgICAgICAgICAgIGNhbmQgPSAoYlswXSArIG9mZlswXSwgYlsxXSArIG9mZlsxXSkKICAgICAgICAgICAgaWYgY2FuZCBpbiBjZWxsczoKICAgICAgICAgICAgICAgIG54dCA9IGNhbmQKICAgICAgICAgICAgICAgIGJhY2sgPSBfQ1dbKGlkeCArIGsgLSAxKSAlIDhdCiAgICAgICAgICAgICAgICBuZXdfcHJldiA9IChiWzBdICsgYmFja1swXSwgYlsxXSArIGJhY2tbMV0pCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGlmIG54dCBpcyBOb25lOgogICAgICAgICAgICBicmVhawogICAgICAgIGlmIHNlY29uZCBpcyBOb25lOgogICAgICAgICAgICBzZWNvbmQgPSBueHQKICAgICAgICBlbGlmIGIgPT0gc3RhcnQgYW5kIG54dCA9PSBzZWNvbmQ6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgY29udG91ci5hcHBlbmQobnh0KQogICAgICAgIHByZXYsIGIgPSBuZXdfcHJldiwgbnh0CiAgICBpZiBsZW4oY29udG91cikgPiAxIGFuZCBjb250b3VyWy0xXSA9PSBjb250b3VyWzBdOgogICAgICAgIGNvbnRvdXIucG9wKCkKICAgIHJldHVybiBjb250b3VyCgoKZGVmIF9jb3JuZXJfcG9pbnRzKGNvbnRvdXIpOgogICAgIiIiUmVkdWNlIGNvbnRvdXIgdG8gZGlyZWN0aW9uLWNoYW5nZSBwb2ludHMuIiIiCiAgICBpZiBsZW4oY29udG91cikgPD0gMjoKICAgICAgICByZXR1cm4gbGlzdChjb250b3VyKQogICAgbSA9IGxlbihjb250b3VyKQogICAgY29ybmVycyA9IFtdCiAgICBmb3IgaSBpbiByYW5nZShtKToKICAgICAgICBwcmV2X2MsIGN1ciwgbnh0ID0gY29udG91cltpIC0gMV0sIGNvbnRvdXJbaV0sIGNvbnRvdXJbKGkgKyAxKSAlIG1dCiAgICAgICAgZF9pbiA9IChjdXJbMF0gLSBwcmV2X2NbMF0sIGN1clsxXSAtIHByZXZfY1sxXSkKICAgICAgICBkX291dCA9IChueHRbMF0gLSBjdXJbMF0sIG54dFsxXSAtIGN1clsxXSkKICAgICAgICBpZiBkX2luICE9IGRfb3V0OgogICAgICAgICAgICBjb3JuZXJzLmFwcGVuZChjdXIpCiAgICByZXR1cm4gY29ybmVycwoKCmRlZiBfb2JqZWN0X2hhc2goY2VsbHMsIGNvbG9yKToKICAgICIiIlRyYW5zbGF0aW9uLWludmFyaWFudCBzaWduYXR1cmUgb2YgYW4gb2JqZWN0LiIiIgogICAgbWluX3IgPSBtaW4ociBmb3IgciwgXyBpbiBjZWxscykKICAgIG1pbl9jID0gbWluKGMgZm9yIF8sIGMgaW4gY2VsbHMpCiAgICBub3JtID0gc29ydGVkKChyIC0gbWluX3IsIGMgLSBtaW5fYykgZm9yIHIsIGMgaW4gY2VsbHMpCiAgICBwYXlsb2FkID0gcmVwcigoY29sb3IsIG5vcm0pKS5lbmNvZGUoKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMShwYXlsb2FkKS5oZXhkaWdlc3QoKVs6MTZdCgoKZGVmIHNlZ21lbnRfZ3JpZChncmlkOiBTZXF1ZW5jZVtTZXF1ZW5jZVtpbnRdXSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJTZWdtZW50IGEgZ3JpZCBpbnRvIGNvbm5lY3RlZCBjb21wb25lbnRzLgogICAgCiAgICBSZXR1cm5zIGRpY3Qgd2l0aDoKICAgICAgLSBub2RlczogbGlzdCBvZiB7aWQsIGNvbG9yLCBoYXNoLCBwaXhlbHMsIGJvdW5kYXJ5LCBjaGlsZHJlbn0KICAgICAgLSBhZGphY2VuY3lfbGlzdDogbGlzdCBvZiBbaSwgal0gcGFpcnMKICAgICIiIgogICAgaGVpZ2h0ID0gbGVuKGdyaWQpCiAgICB3aWR0aCA9IGxlbihncmlkWzBdKSBpZiBoZWlnaHQgZWxzZSAwCiAgICBpZiBub3QgaGVpZ2h0IG9yIG5vdCB3aWR0aDoKICAgICAgICByZXR1cm4geyJub2RlcyI6IFtdLCAiYWRqYWNlbmN5X2xpc3QiOiBbXX0KICAgIAogICAgIyBDb25uZWN0ZWQgY29tcG9uZW50cyB2aWEgZmxvb2QgZmlsbAogICAgY29tcF9pZCA9IFtbLTFdICogd2lkdGggZm9yIF8gaW4gcmFuZ2UoaGVpZ2h0KV0KICAgIGNvbXBvbmVudHMgPSBbXQogICAgCiAgICBmb3Igc3IgaW4gcmFuZ2UoaGVpZ2h0KToKICAgICAgICBmb3Igc2MgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICBpZiBjb21wX2lkW3NyXVtzY10gIT0gLTE6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB2YWx1ZSA9IGdyaWRbc3JdW3NjXQogICAgICAgICAgICBjaWQgPSBsZW4oY29tcG9uZW50cykKICAgICAgICAgICAgY2VsbHMgPSBzZXQoKQogICAgICAgICAgICBzdGFjayA9IFsoc3IsIHNjKV0KICAgICAgICAgICAgd2hpbGUgc3RhY2s6CiAgICAgICAgICAgICAgICByLCBjID0gc3RhY2sucG9wKCkKICAgICAgICAgICAgICAgIGlmIChyLCBjKSBpbiBjZWxsczoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgaWYgMCA8PSByIDwgaGVpZ2h0IGFuZCAwIDw9IGMgPCB3aWR0aCBhbmQgZ3JpZFtyXVtjXSA9PSB2YWx1ZSBhbmQgY29tcF9pZFtyXVtjXSA9PSAtMToKICAgICAgICAgICAgICAgICAgICBjZWxscy5hZGQoKHIsIGMpKQogICAgICAgICAgICAgICAgICAgIGNvbXBfaWRbcl1bY10gPSBjaWQKICAgICAgICAgICAgICAgICAgICBmb3IgZHIsIGRjIGluIF9PUlRIOgogICAgICAgICAgICAgICAgICAgICAgICBuciwgbmMgPSByICsgZHIsIGMgKyBkYwogICAgICAgICAgICAgICAgICAgICAgICBpZiAwIDw9IG5yIDwgaGVpZ2h0IGFuZCAwIDw9IG5jIDwgd2lkdGggYW5kIGNvbXBfaWRbbnJdW25jXSA9PSAtMSBhbmQgZ3JpZFtucl1bbmNdID09IHZhbHVlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RhY2suYXBwZW5kKChuciwgbmMpKQogICAgICAgICAgICBjb21wb25lbnRzLmFwcGVuZCh7InZhbHVlIjogdmFsdWUsICJjZWxscyI6IGNlbGxzLCAic3RhcnQiOiAoc3IsIHNjKX0pCiAgICAKICAgICMgQnVpbGQgbm9kZXMKICAgIG5vZGVzID0gW10KICAgIGZvciBjaWQsIGNvbXAgaW4gZW51bWVyYXRlKGNvbXBvbmVudHMpOgogICAgICAgIGNvbG9yX2NoYXIgPSBBUkNfQ09MT1JfQ0hBUlNbbWF4KDAsIG1pbigxNSwgY29tcFsidmFsdWUiXSkpXQogICAgICAgIG9ial9oYXNoID0gX29iamVjdF9oYXNoKGNvbXBbImNlbGxzIl0sIGNvbXBbInZhbHVlIl0pCiAgICAgICAgY29udG91ciA9IF90cmFjZV9vdXRlcl9jb250b3VyKGNvbXBbImNlbGxzIl0sIGNvbXBbInN0YXJ0Il0pCiAgICAgICAgYm91bmRhcnkgPSBfY29ybmVyX3BvaW50cyhjb250b3VyKQogICAgICAgIG5vZGVzLmFwcGVuZCh7CiAgICAgICAgICAgICJpZCI6IGNpZCwKICAgICAgICAgICAgImNvbG9yIjogY29sb3JfY2hhciwKICAgICAgICAgICAgImhhc2giOiBvYmpfaGFzaCwKICAgICAgICAgICAgInBpeGVscyI6IGxlbihjb21wWyJjZWxscyJdKSwKICAgICAgICAgICAgImJvdW5kYXJ5IjogW1tyLCBjXSBmb3IgciwgYyBpbiBib3VuZGFyeVs6MTJdXSwgICMgbGltaXQgc2l6ZQogICAgICAgICAgICAiY2hpbGRyZW4iOiBbXSwKICAgICAgICB9KQogICAgCiAgICAjIENvbXB1dGUgY29udGFpbm1lbnQgKGNoaWxkcmVuKQogICAgZm9yIG91dGVyX2lkLCBvdXRlcl9jb21wIGluIGVudW1lcmF0ZShjb21wb25lbnRzKToKICAgICAgICBvdXRlcl9jZWxscyA9IG91dGVyX2NvbXBbImNlbGxzIl0KICAgICAgICBmb3IgaW5uZXJfaWQsIGlubmVyX2NvbXAgaW4gZW51bWVyYXRlKGNvbXBvbmVudHMpOgogICAgICAgICAgICBpZiBpbm5lcl9pZCA9PSBvdXRlcl9pZDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgQ2hlY2sgaWYgaW5uZXIgaXMgZnVsbHkgZW5jbG9zZWQgYnkgb3V0ZXIKICAgICAgICAgICAgaW5uZXJfY2VsbHMgPSBpbm5lcl9jb21wWyJjZWxscyJdCiAgICAgICAgICAgIGlmIGFsbChhbnkoKHIgKyBkciwgYyArIGRjKSBpbiBvdXRlcl9jZWxscyBmb3IgZHIsIGRjIGluIF9PUlRIKQogICAgICAgICAgICAgICAgICAgZm9yIHIsIGMgaW4gaW5uZXJfY2VsbHMpOgogICAgICAgICAgICAgICAgIyBDaGVjayBubyBwYXRoIGZyb20gaW5uZXIgdG8gZ3JpZCBlZGdlIHdpdGhvdXQgY3Jvc3Npbmcgb3V0ZXIKICAgICAgICAgICAgICAgICMgU2ltcGxlIGNoZWNrOiBhbGwgbmVpZ2hib3JzIG9mIGlubmVyIGNlbGxzIGFyZSBlaXRoZXIgaW5uZXIgb3Igb3V0ZXIKICAgICAgICAgICAgICAgIGlzX2NoaWxkID0gVHJ1ZQogICAgICAgICAgICAgICAgZm9yIHIsIGMgaW4gaW5uZXJfY2VsbHM6CiAgICAgICAgICAgICAgICAgICAgZm9yIGRyLCBkYyBpbiBfT1JUSDoKICAgICAgICAgICAgICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKICAgICAgICAgICAgICAgICAgICAgICAgaWYgMCA8PSBuciA8IGhlaWdodCBhbmQgMCA8PSBuYyA8IHdpZHRoOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgY29tcF9pZFtucl1bbmNdICE9IG91dGVyX2lkIGFuZCBjb21wX2lkW25yXVtuY10gIT0gaW5uZXJfaWQ6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaXNfY2hpbGQgPSBGYWxzZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IGlzX2NoaWxkOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgaWYgaXNfY2hpbGQ6CiAgICAgICAgICAgICAgICAgICAgbm9kZXNbb3V0ZXJfaWRdWyJjaGlsZHJlbiJdLmFwcGVuZChpbm5lcl9pZCkKICAgIAogICAgIyBDb21wdXRlIGFkamFjZW5jeQogICAgYWRqX3NldCA9IHNldCgpCiAgICBmb3IgciBpbiByYW5nZShoZWlnaHQpOgogICAgICAgIGZvciBjIGluIHJhbmdlKHdpZHRoKToKICAgICAgICAgICAgY2lkID0gY29tcF9pZFtyXVtjXQogICAgICAgICAgICBmb3IgZHIsIGRjIGluIF9PUlRIOgogICAgICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKICAgICAgICAgICAgICAgIGlmIDAgPD0gbnIgPCBoZWlnaHQgYW5kIDAgPD0gbmMgPCB3aWR0aDoKICAgICAgICAgICAgICAgICAgICBuaWQgPSBjb21wX2lkW25yXVtuY10KICAgICAgICAgICAgICAgICAgICBpZiBjaWQgIT0gbmlkOgogICAgICAgICAgICAgICAgICAgICAgICBwYWlyID0gKG1pbihjaWQsIG5pZCksIG1heChjaWQsIG5pZCkpCiAgICAgICAgICAgICAgICAgICAgICAgIGFkal9zZXQuYWRkKHBhaXIpCiAgICAKICAgIHJldHVybiB7CiAgICAgICAgIm5vZGVzIjogbm9kZXMsCiAgICAgICAgImFkamFjZW5jeV9saXN0Ijogc29ydGVkKGFkal9zZXQpLAogICAgfQoKCmRlZiBjb21wYWN0X3NlZ21lbnRhdGlvbl9zdW1tYXJ5KHNlZzogZGljdCwgbWF4X25vZGVzOiBpbnQgPSAyMCkgLT4gc3RyOgogICAgIiIiQ3JlYXRlIGEgY29tcGFjdCB0ZXh0IHN1bW1hcnkgb2Ygc2VnbWVudGF0aW9uIGZvciBMTE0gY29udGV4dC4iIiIKICAgIG5vZGVzID0gc2VnLmdldCgibm9kZXMiLCBbXSkKICAgIGFkaiA9IHNlZy5nZXQoImFkamFjZW5jeV9saXN0IiwgW10pCiAgICAKICAgIGxpbmVzID0gW2YiT2JqZWN0czoge2xlbihub2Rlcyl9LCBBZGphY2VuY2llczoge2xlbihhZGopfSJdCiAgICBmb3Igbm9kZSBpbiBub2Rlc1s6bWF4X25vZGVzXToKICAgICAgICBib3VuZGFyeV9zdHIgPSAiLCIuam9pbihmIntyfSx7Y30iIGZvciByLCBjIGluIG5vZGVbImJvdW5kYXJ5Il1bOjRdKQogICAgICAgIGNoaWxkcmVuX3N0ciA9IGYiLCBjaGlsZHJlbj1beycsJy5qb2luKG1hcChzdHIsIG5vZGVbJ2NoaWxkcmVuJ10pKX1dIiBpZiBub2RlWyJjaGlsZHJlbiJdIGVsc2UgIiIKICAgICAgICBsaW5lcy5hcHBlbmQoCiAgICAgICAgICAgIGYiICBvYmoje25vZGVbJ2lkJ119OiBjb2xvcj17bm9kZVsnY29sb3InXX0sIHNpemU9e25vZGVbJ3BpeGVscyddfXB4LCAiCiAgICAgICAgICAgIGYiaGFzaD17bm9kZVsnaGFzaCddWzo4XX0sIGNvcm5lcnM9W3tib3VuZGFyeV9zdHJ9XXtjaGlsZHJlbl9zdHJ9IgogICAgICAgICkKICAgIGlmIGxlbihub2RlcykgPiBtYXhfbm9kZXM6CiAgICAgICAgbGluZXMuYXBwZW5kKGYiICAuLi4gYW5kIHtsZW4obm9kZXMpIC0gbWF4X25vZGVzfSBtb3JlIG9iamVjdHMiKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykK").decode("utf-8")
(WORKING / "grid_utils.py").write_text(grid_utils_src)

# Goal classifier
goal_classifier_src = base64.b64decode("...").decode("utf-8")
(WORKING / "goal_classifier.py").write_text(goal_classifier_src)

# LLM policy v7
llm_policy_src = base64.b64decode("...").decode("utf-8")
(WORKING / "llm_policy_v7.py").write_text(llm_policy_src)

# LLM agent adapter
llm_agent_src = base64.b64decode("IiIiQVJDLUFHSS0zIEFnZW50IGFkYXB0ZXIgZm9yIHRoZSBMTE0tYmFzZWQgcG9saWN5LgoKUGx1Z3MgTExNUG9saWN5IGludG8gdGhlIEFSQy1BR0ktMy1BZ2VudHMgcmVmZXJlbmNlIGZyYW1ld29yay4KVGhlIGFnZW50IGNhbGxzIHRoZSBMTE0gdG8gZ2V0IGFjdGlvbnMsIHdpdGggYW50aS1sb29wIGRldGVjdGlvbgphbmQgYWN0aW9uIGJhdGNoaW5nIGZvciBlZmZpY2llbmN5LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSB0aHJlYWRpbmcgaW1wb3J0IFJMb2NrCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCiMgRW5zdXJlIGltcG9ydHMgd29yayBib3RoIGluIGNvbXBldGl0aW9uIGFuZCBsb2NhbGx5Cl9kaXIgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKX3BhcmVudCA9IG9zLnBhdGguZGlybmFtZShfZGlyKQpmb3IgcCBpbiAoX2RpciwgX3BhcmVudCk6CiAgICBpZiBwIG5vdCBpbiBzeXMucGF0aDoKICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgcCkKCmZyb20gYXJjZW5naW5lIGltcG9ydCBBY3Rpb25JbnB1dCwgRnJhbWVEYXRhLCBHYW1lQWN0aW9uLCBHYW1lU3RhdGUKZnJvbSBhZ2VudHMuYWdlbnQgaW1wb3J0IEFnZW50Cgp0cnk6CiAgICBmcm9tIC5sbG1fcG9saWN5IGltcG9ydCBMTE1Qb2xpY3ksIExMTUNvbmZpZywgRnJhbWVJbmZvCiAgICBmcm9tIC5ncmlkX3V0aWxzIGltcG9ydCBmb3JtYXRfZ3JpZF9hc2NpaQpleGNlcHQgSW1wb3J0RXJyb3I6CiAgICBmcm9tIGxsbV9wb2xpY3kgaW1wb3J0IExMTVBvbGljeSwgTExNQ29uZmlnLCBGcmFtZUluZm8KICAgIGZyb20gZ3JpZF91dGlscyBpbXBvcnQgZm9ybWF0X2dyaWRfYXNjaWkKCgpkZWYgX2V4dHJhY3RfZ3JpZChmcmFtZV9kYXRhOiBGcmFtZURhdGEpIC0+IHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXToKICAgICIiIkV4dHJhY3QgdGhlIG51bWVyaWMgZ3JpZCBmcm9tIGEgRnJhbWVEYXRhIG9iamVjdC4iIiIKICAgICMgVHJ5IG11bHRpcGxlIHBhdGhzIHRvIGdldCB0aGUgZ3JpZCBkYXRhCiAgICBmb3IgYXR0cl9wYXRoIGluIFsKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUuZGF0YSwKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUucGxhbmVzWzBdIGlmIGYucmF3X3N0YXRlLnBsYW5lcyBlbHNlIE5vbmUsCiAgICAgICAgbGFtYmRhIGY6IGYucGxhbmVzWzBdIGlmIGhhc2F0dHIoZiwgJ3BsYW5lcycpIGFuZCBmLnBsYW5lcyBlbHNlIE5vbmUsCiAgICBdOgogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0YSA9IGF0dHJfcGF0aChmcmFtZV9kYXRhKQogICAgICAgICAgICBpZiBkYXRhIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBoYXNhdHRyKGRhdGEsICd0b2xpc3QnKToKICAgICAgICAgICAgICAgIHJvd3MgPSBkYXRhLnRvbGlzdCgpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICByb3dzID0gbGlzdChkYXRhKQogICAgICAgICAgICBpZiByb3dzIGFuZCByb3dzWzBdOgogICAgICAgICAgICAgICAgcmV0dXJuIHR1cGxlKHR1cGxlKGludChjKSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gcm93cykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBjb250aW51ZQogICAgIyBBYnNvbHV0ZSBmYWxsYmFjawogICAgcmV0dXJuICgoMCwpICogNjQsKSAqIDY0CgoKZGVmIF9mcmFtZV90b19pbmZvKGZyYW1lX2RhdGE6IEZyYW1lRGF0YSkgLT4gRnJhbWVJbmZvOgogICAgIiIiQ29udmVydCBGcmFtZURhdGEgdG8gb3VyIEZyYW1lSW5mby4iIiIKICAgIGdyaWQgPSBfZXh0cmFjdF9ncmlkKGZyYW1lX2RhdGEpCiAgICAKICAgICMgU3RhdGUgbmFtZQogICAgdHJ5OgogICAgICAgIHN0YXRlID0gZnJhbWVfZGF0YS5zdGF0ZQogICAgICAgIHN0YXRlX25hbWUgPSBzdGF0ZS5uYW1lIGlmIGhhc2F0dHIoc3RhdGUsICduYW1lJykgZWxzZSBzdHIoc3RhdGUpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHN0YXRlX25hbWUgPSAiUExBWUlORyIKICAgIAogICAgIyBMZXZlbHMgY29tcGxldGVkCiAgICB0cnk6CiAgICAgICAgbGV2ZWxzX2NvbXBsZXRlZCA9IGludChmcmFtZV9kYXRhLmxldmVsc19jb21wbGV0ZWQpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGxldmVsc19jb21wbGV0ZWQgPSAwCiAgICAKICAgICMgQXZhaWxhYmxlIGFjdGlvbnMKICAgIGF2YWlsYWJsZSA9IFtdCiAgICB0cnk6CiAgICAgICAgZm9yIGFjdF9pZCBpbiBmcmFtZV9kYXRhLmF2YWlsYWJsZV9hY3Rpb25zOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBhY3QgPSBHYW1lQWN0aW9uLmZyb21faWQoaW50KGFjdF9pZCkpCiAgICAgICAgICAgICAgICBpZiBhY3QubmFtZSAhPSAiUkVTRVQiOgogICAgICAgICAgICAgICAgICAgIGF2YWlsYWJsZS5hcHBlbmQoYWN0Lm5hbWUpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGF2YWlsYWJsZSA9IFsiQUNUSU9OMSIsICJBQ1RJT04yIiwgIkFDVElPTjMiLCAiQUNUSU9ONCIsICJBQ1RJT041IiwgIkFDVElPTjYiLCAiQUNUSU9ONyJdCiAgICAKICAgICMgU3RlcCBudW1iZXIKICAgIHRyeToKICAgICAgICBzdGVwID0gaW50KGZyYW1lX2RhdGEuc3RlcCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgc3RlcCA9IDAKICAgIAogICAgcmV0dXJuIEZyYW1lSW5mbygKICAgICAgICBncmlkPWdyaWQsCiAgICAgICAgc3RlcD1zdGVwLAogICAgICAgIGxldmVsPWxldmVsc19jb21wbGV0ZWQgKyAxLAogICAgICAgIHN0YXRlX25hbWU9c3RhdGVfbmFtZSwKICAgICAgICBsZXZlbHNfY29tcGxldGVkPWxldmVsc19jb21wbGV0ZWQsCiAgICAgICAgYXZhaWxhYmxlX2FjdGlvbnM9YXZhaWxhYmxlLAogICAgKQoKCmNsYXNzIExMTUFnZW50KEFnZW50KToKICAgICIiIkxMTS1wb3dlcmVkIGFnZW50IGZvciBBUkMtQUdJLTMuIiIiCiAgICAKICAgIF9BQ1RJT05fTE9DSyA9IFJMb2NrKCkKICAgIAogICAgQUNUSU9OX0JVREdFVCA9IDQwMAogICAgTUFYX0FDVElPTlMgPSBBQ1RJT05fQlVER0VUCiAgICAKICAgICMgU29mdCB0aW1lIGxpbWl0IHBlciBnYW1lIChzZWNvbmRzKQogICAgIyAxMTAgZ2FtZXMgw5cgNCBtaW4gPSA0NDAgbWluIOKJiCA3LjMgaG91cnMg4oCUIGZpdHMgaW4gOWggd2l0aCBtYXJnaW4KICAgIEdBTUVfVElNRV9MSU1JVCA9IDI0MAogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCphcmdzLCAqKmt3YXJncykKICAgICAgICBzZWxmLmNvbmZpZyA9IExMTUNvbmZpZy5mcm9tX2VudigpCiAgICAgICAgc2VsZi5wb2xpY3kgPSBMTE1Qb2xpY3koc2VsZi5jb25maWcpCiAgICAgICAgc2VsZi5faG9sZHNfYWN0aW9uX2xvY2sgPSBGYWxzZQogICAgICAgIHNlbGYuX2dhbWVfc3RhcnRfdGltZSA9IHRpbWUudGltZSgpCiAgICAgICAgc2VsZi5fYWN0aW9uc19pbl9nYW1lID0gMAogICAgICAgIHNlbGYuX2ZpcnN0X2FjdGlvbiA9IFRydWUKICAgIAogICAgQHByb3BlcnR5CiAgICBkZWYgbmFtZShzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIGYie3N1cGVyKCkubmFtZX0ubGxtLXYyIgogICAgCiAgICBkZWYgaXNfZG9uZShzZWxmLCBmcmFtZXM6IGxpc3RbRnJhbWVEYXRhXSwgbGF0ZXN0X2ZyYW1lOiBGcmFtZURhdGEpIC0+IGJvb2w6CiAgICAgICAgIiIiU3RvcCBjb25kaXRpb25zOiB3aW4sIGJ1ZGdldCwgb3IgdGltZSBsaW1pdC4iIiIKICAgICAgICBpZiBzZWxmLl9hY3Rpb25zX2luX2dhbWUgPj0gc2VsZi5BQ1RJT05fQlVER0VUOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgbGF0ZXN0X2ZyYW1lLnN0YXRlIGlzIEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICBpZiB0aW1lLnRpbWUoKSAtIHNlbGYuX2dhbWVfc3RhcnRfdGltZSA+IHNlbGYuR0FNRV9USU1FX0xJTUlUOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHJldHVybiBGYWxzZQogICAgCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3RvX2dhbWVfYWN0aW9uKHByb3Bvc2FsOiBkaWN0KSAtPiBHYW1lQWN0aW9uOgogICAgICAgICIiIkNvbnZlcnQgcG9saWN5IG91dHB1dCB0byBHYW1lQWN0aW9uIGVudW0uIiIiCiAgICAgICAgYWN0aW9uX25hbWUgPSBwcm9wb3NhbC5nZXQoIm5hbWUiLCAiUkVTRVQiKQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgYWN0aW9uID0gR2FtZUFjdGlvblthY3Rpb25fbmFtZV0KICAgICAgICBleGNlcHQgS2V5RXJyb3I6CiAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICAKICAgICAgICBpZiBhY3Rpb24uaXNfY29tcGxleCgpOgogICAgICAgICAgICB4ID0gcHJvcG9zYWwuZ2V0KCJ4IiwgMCkgb3IgMAogICAgICAgICAgICB5ID0gcHJvcG9zYWwuZ2V0KCJ5IiwgMCkgb3IgMAogICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoeyJ4IjogbWF4KDAsIG1pbig2MywgaW50KHgpKSksICJ5IjogbWF4KDAsIG1pbig2MywgaW50KHkpKSl9KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICAKICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0gcHJvcG9zYWwuZ2V0KCJyZWFzb25pbmciLCB7fSkKICAgICAgICByZXR1cm4gYWN0aW9uCiAgICAKICAgIGRlZiBjaG9vc2VfYWN0aW9uKAogICAgICAgIHNlbGYsIGZyYW1lczogbGlzdFtGcmFtZURhdGFdLCBsYXRlc3RfZnJhbWU6IEZyYW1lRGF0YQogICAgKSAtPiBHYW1lQWN0aW9uOgogICAgICAgICIiIk1haW4gZW50cnkgcG9pbnQgY2FsbGVkIGJ5IHRoZSBmcmFtZXdvcmsgZWFjaCB0dXJuLiIiIgogICAgICAgIHNlbGYuX0FDVElPTl9MT0NLLmFjcXVpcmUoKQogICAgICAgIHNlbGYuX2hvbGRzX2FjdGlvbl9sb2NrID0gVHJ1ZQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgZnJhbWVfaW5mbyA9IF9mcmFtZV90b19pbmZvKGxhdGVzdF9mcmFtZSkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgSW5pdGlhbGl6ZSBvbiBmaXJzdCBjYWxsCiAgICAgICAgICAgIGlmIHNlbGYuX2ZpcnN0X2FjdGlvbjoKICAgICAgICAgICAgICAgIHNlbGYuX2ZpcnN0X2FjdGlvbiA9IEZhbHNlCiAgICAgICAgICAgICAgICBzZWxmLl9nYW1lX3N0YXJ0X3RpbWUgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgc2VsZi5wb2xpY3kucmVzZXRfZ2FtZSgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEF1dG8tcmVzZXQgb24gR0FNRV9PVkVSCiAgICAgICAgICAgIGlmIGZyYW1lX2luZm8uc3RhdGVfbmFtZSA9PSAiR0FNRV9PVkVSIjoKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBzZWxmLnBvbGljeS5ub3RlX2dhbWVfb3ZlcihmcmFtZV9pbmZvKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBwYXNzCiAgICAgICAgICAgICAgICBhY3Rpb24gPSBHYW1lQWN0aW9uLlJFU0VUCiAgICAgICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoe30pCiAgICAgICAgICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0geyJhdXRvX3Jlc2V0IjogVHJ1ZSwgImdhbWVfb3ZlciI6IFRydWV9CiAgICAgICAgICAgICAgICByZXR1cm4gYWN0aW9uCiAgICAgICAgICAgIAogICAgICAgICAgICAjIENhbGwgdGhlIExMTSBwb2xpY3kKICAgICAgICAgICAgcHJvcG9zYWwgPSBzZWxmLnBvbGljeS5jaG9vc2VfYWN0aW9uKGZyYW1lX2luZm8pCiAgICAgICAgICAgIHNlbGYuX2FjdGlvbnNfaW5fZ2FtZSArPSAxCiAgICAgICAgICAgIAogICAgICAgICAgICByZXR1cm4gc2VsZi5fdG9fZ2FtZV9hY3Rpb24ocHJvcG9zYWwpCiAgICAgICAgICAgIAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgaW1wb3J0IHRyYWNlYmFjawogICAgICAgICAgICBwcmludChmIltMTE1BZ2VudF0gRXJyb3I6IHtlfSIpCiAgICAgICAgICAgIHRyYWNlYmFjay5wcmludF9leGMoKQogICAgICAgICAgICBhY3Rpb24gPSBHYW1lQWN0aW9uLlJFU0VUCiAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICAgICAgYWN0aW9uLnJlYXNvbmluZyA9IHsiZXJyb3IiOiBzdHIoZSl9CiAgICAgICAgICAgIHJldHVybiBhY3Rpb24KICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBpZiBzZWxmLl9ob2xkc19hY3Rpb25fbG9jazoKICAgICAgICAgICAgICAgIHNlbGYuX2hvbGRzX2FjdGlvbl9sb2NrID0gRmFsc2UKICAgICAgICAgICAgICAgIHNlbGYuX0FDVElPTl9MT0NLLnJlbGVhc2UoKQogICAgCiAgICBkZWYgdGFrZV9hY3Rpb24oc2VsZiwgYWN0aW9uOiBHYW1lQWN0aW9uKTogICMgdHlwZTogaWdub3JlW292ZXJyaWRlXQogICAgICAgICIiIkV4ZWN1dGUgYWN0aW9uIGFuZCByZWNvcmQgdGhlIHJlc3VsdC4iIiIKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyYW1lID0gc3VwZXIoKS50YWtlX2FjdGlvbihhY3Rpb24pCiAgICAgICAgICAgIGlmIGZyYW1lIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgZnJhbWUuYWN0aW9uX2lucHV0ID0gQWN0aW9uSW5wdXQoCiAgICAgICAgICAgICAgICAgICAgaWQ9YWN0aW9uLAogICAgICAgICAgICAgICAgICAgIGRhdGE9YWN0aW9uLmFjdGlvbl9kYXRhLm1vZGVsX2R1bXAoKSwKICAgICAgICAgICAgICAgICAgICByZWFzb25pbmc9Z2V0YXR0cihhY3Rpb24sICJyZWFzb25pbmciLCBOb25lKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgcmV0dXJuIGZyYW1lCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaWYgZ2V0YXR0cihzZWxmLCAiX2hvbGRzX2FjdGlvbl9sb2NrIiwgRmFsc2UpOgogICAgICAgICAgICAgICAgc2VsZi5faG9sZHNfYWN0aW9uX2xvY2sgPSBGYWxzZQogICAgICAgICAgICAgICAgc2VsZi5fQUNUSU9OX0xPQ0sucmVsZWFzZSgpCiAgICAKICAgIGRlZiBmaW5hbGl6ZV9kaWFnbm9zdGljcyhzZWxmKSAtPiBOb25lOgogICAgICAgICIiIkxvZyBmaW5hbCBzdGF0cyBmb3IgdGhpcyBnYW1lLiIiIgogICAgICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHNlbGYuX2dhbWVfc3RhcnRfdGltZQogICAgICAgIHByaW50KGYiW0xMTUFnZW50XSBHYW1lIGRvbmU6ICIKICAgICAgICAgICAgICBmIntzZWxmLl9hY3Rpb25zX2luX2dhbWV9IGFjdGlvbnMgaW4ge2VsYXBzZWQ6LjBmfXMsICIKICAgICAgICAgICAgICBmIntzZWxmLnBvbGljeS50b3RhbF90b2tlbnN9IHRva2VucyAoe3NlbGYucG9saWN5LnRvdGFsX2NhbGxzfSBjYWxscywgIgogICAgICAgICAgICAgIGYie3NlbGYucG9saWN5LmNsaWVudC50b3RhbF90aW1lOi4wZn1zIExMTSB0aW1lKSwgIgogICAgICAgICAgICAgIGYie2xlbihzZWxmLnBvbGljeS50cmFja2VyLnVuaXF1ZV9zdGF0ZXNfc2Vlbil9IHVuaXF1ZSBzdGF0ZXMsICIKICAgICAgICAgICAgICBmIntzZWxmLnBvbGljeS50cmFja2VyLmdhbWVfb3Zlcl9jb3VudH0gZGVhdGhzIikK").decode("utf-8")
(WORKING / "llm_agent.py").write_text(llm_agent_src)

print("✓ Agent code deployed")


In [ ]:
# Import and configure
sys.path.insert(0, str(WORKING))

from llm_policy_v7 import LLMPolicyV7, LLMConfig
from llm_agent import LLMAgent

# Configure LLM
config = LLMConfig(
    base_url="http://127.0.0.1:8000/v1",
    model_id="qwen3.6-27b-fp8",
    api_key="EMPTY",
    max_output_tokens=2048,
    temperature=0.6,
    enable_thinking=True,
)

print(f"LLM config: {config.base_url} / {config.model_id}")
print("✓ LLM policy v7 ready")


In [ ]:
# Run agent on competition games
import arcengine
from pathlib import Path

games_dir = Path("/kaggle/input/arc-prize-2026-arc-agi-3/games")
games = sorted(games_dir.glob("*.game"))
print(f"Found {len(games)} games")

results = []
for game_path in games:
    print(f"\n{'='*60}")
    print(f"Game: {game_path.name}")
    print('='*60)
    
    # Load game
    with open(game_path) as f:
        game_data = json.load(f)
    
    env = arcengine.Environment(game_data)
    policy = LLMPolicyV7(config)
    
    # Run game
    max_actions = 400
    for step in range(max_actions):
        frame = env.render()
        
        # Check if game is over
        if frame.state_name in ("WIN", "GAME_OVER"):
            break
        
        # Check if should abandon
        if policy.should_abandon():
            print(f"  [ABANDON] 0 levels after {policy.total_tokens} tokens")
            break
        
        # Get action from policy
        action = policy.choose_action(frame)
        
        # FIX BUG 2: Handle ABANDON action (early abandon)
        # ABANDON means policy wants to stop this game and move to next
        # (not RESET which restarts in same server slot)
        if action.get("name") == "ABANDON":
            print(f"  [ABANDON] 0 levels after {policy.total_tokens} tokens, {step+1} actions")
            break
        
        # Execute action
        env.step(action["name"])
        
        # Progress indicator
        if step % 50 == 0:
            print(f"  Step {step} | Level {frame.level+1} | Tokens: {policy.total_tokens}")
    
    # Record result
    final_frame = env.render()
    result = {
        "game": game_path.name,
        "levels_completed": final_frame.levels_completed,
        "total_actions": step + 1,
        "total_tokens": policy.total_tokens,
        "abandoned": policy._should_abandon,
    }
    results.append(result)
    
    print(f"  Result: {final_frame.levels_completed} levels | {step+1} actions | {policy.total_tokens} tokens")
    
    # Reset for next game
    policy.reset_game()
    env.reset()

print(f"\n{'='*60}")
print(f"Completed {len(results)} games")
print('='*60)


In [ ]:
# Generate submission.parquet
print("Generating submission.parquet...")

submission = {
    "game_id": [r["game"] for r in results],
    "levels_completed": [r["levels_completed"] for r in results],
    "total_actions": [r["total_actions"] for r in results],
    "total_tokens": [r["total_tokens"] for r in results],
}

import pandas as pd
df = pd.DataFrame(submission)
df.to_parquet(WORKING / "submission.parquet", index=False)

print(f"✓ submission.parquet generated ({len(results)} games)")
print(f"\nSummary:")
print(f"  Games completed: {len(results)}")
print(f"  Games abandoned: {sum(1 for r in results if r['abandoned'])}")
print(f"  Total levels: {sum(r['levels_completed'] for r in results)}")
print(f"  Total actions: {sum(r['total_actions'] for r in results)}")
print(f"  Total tokens: {sum(r['total_tokens'] for r in results)}")

if TRUE_SUBMISSION:
    print("\n✓ TRUE_SUBMISSION mode - submission.parquet ready for Kaggle")
else:
    print("\n⚠ Not in TRUE_SUBMISSION mode - this was a test run")
